In [ ]:
!pip install -q langchain-community langchain-text-splitters langchain-core \
    pypdf pymupdf sentence-transformers chromadb scikit-learn langchain-groq

In [ ]:
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any, Union

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
doc = Document(
    page_content="this is the main content I am using to create RAG",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Harsh Singh",
        "date_created": "2026-10-10",
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Harsh Singh', 'date_created': '2026-10-10'}, page_content='this is the main content I am using to create RAG')

In [ ]:
os.makedirs("../data/text_files", exist_ok=True)

sample_text = {
    "../data/text_files/rag_intro.txt": """Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from external knowledge bases before generating a response. Standard large language models rely only on their fixed training data. This limitation can cause false information (hallucinations) or outdated answers when users ask about private company files or recent events. RAG solves this problem by connecting the model to live external sources like PDFs, databases, or company wikis.

How RAG Works
The RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).
Indexing (Preparation): Documents are split into small sections called chunks. These pieces pass through an embedding model to convert the text into numerical vectors, which are saved in a vector database.
Retrieval (Search): When you type a question, the system turns your query into a vector and searches the vector database for text chunks with a similar meaning.
Augmentation and Generation: The system combines your original question with the retrieved text snippets to form an expanded prompt. The large language model reads this extra context and writes a precise, grounded answer.

Why Use RAG
No Retraining Needed: Updating a model via fine-tuning takes time and money. RAG updates knowledge simply by adding new documents to the database.
Fewer Hallucinations: Grounding the response in real documents ensures the output stays truthful to the provided data.
Data Privacy: Companies can keep private internal files secure while letting an AI assistant search through them safely."""
}

for filepath, content in sample_text.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

print("Sample text files created")

Sample text files created


In [ ]:
# Single-file loader
loader = TextLoader("../data/text_files/rag_intro.txt", encoding="utf-8")
text_documents = loader.load()
print(text_documents)

# Directory loader for .txt files
txt_dir_loader = DirectoryLoader(
    "../data/text_files/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
    use_multithreading=True,
)
txt_docs = txt_dir_loader.load()
txt_docs

[Document(metadata={'source': '../data/text_files/rag_intro.txt'}, page_content="Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from external knowledge bases before generating a response. Standard large language models rely only on their fixed training data. This limitation can cause false information (hallucinations) or outdated answers when users ask about private company files or recent events. RAG solves this problem by connecting the model to live external sources like PDFs, databases, or company wikis.\n\nHow RAG Works\nThe RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).\nIndexing (Preparation): Documents are split into small sections called chunks. These pieces pass through an embedding model to convert the text into numerical vectors, which are saved in a vector database.\nRetrieval (Search): When you type a question, the system t

[Document(metadata={'source': '../data/text_files/rag_intro.txt'}, page_content="Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from external knowledge bases before generating a response. Standard large language models rely only on their fixed training data. This limitation can cause false information (hallucinations) or outdated answers when users ask about private company files or recent events. RAG solves this problem by connecting the model to live external sources like PDFs, databases, or company wikis.\n\nHow RAG Works\nThe RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).\nIndexing (Preparation): Documents are split into small sections called chunks. These pieces pass through an embedding model to convert the text into numerical vectors, which are saved in a vector database.\nRetrieval (Search): When you type a question, the system t

In [ ]:
def process_all_pdfs(pdf_directory: str) -> List[Document]:
    all_documents = []
    pdf_dir = Path(pdf_directory)

    if not pdf_dir.exists():
        print(f"Directory {pdf_directory} does not exist. Creating it — add PDFs there and re-run.")
        pdf_dir.mkdir(parents=True, exist_ok=True)
        return all_documents

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdf")

Found 0 pdf files in ../data/pdf

Total documents loaded: 0


In [ ]:
def split_documents(documents: List[Document], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Document]:
    if not documents:
        print("No documents to split.")
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


# Combine PDF chunks + the sample text docs so you have something to embed
# even if you haven't added PDFs to ../data/pdf yet
source_documents = all_pdf_documents if all_pdf_documents else txt_docs
chunks = split_documents(source_documents)

Split 1 documents into 3 chunks

Example chunk:
Content: Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from external knowledge bases before generating a response...
Metadata: {'source': '../data/text_files/rag_intro.txt'}


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: Union[List[str], str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")

        if isinstance(texts, str):
            texts = [texts]

        if not texts:
            raise ValueError("No texts provided for embedding generation.")

        print(f"Generating embeddings for {len(texts)} texts...")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_sentence_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_1677/3093664550.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
class VectorStore:
    """Handles vector database operations using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"})
            print(f"Vector store initialized successfully. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if not documents:
            print("No documents to add — skipping.")
            return
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")
        ids, metadatas, documents_text, embeddings_list = [], [], [], []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized successfully. Collection: pdf_documents
Existing documents in collection: 3


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
if chunks:
    texts = [doc.page_content for doc in chunks]
    embeddings = embedding_manager.generate_embeddings(texts)
    vectorstore.add_documents(chunks, embeddings)
else:
    print("No chunks available. Add PDFs to ../data/pdf and re-run cells 6-7, or check the sample text loaded.")

Generating embeddings for 3 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (3, 384)
Adding 3 documents to vector store...
Successfully added 3 documents to vector store
Total documents in collection: 6


In [ ]:
class RAGRetriever:
    """Retrieves relevant chunks from the vector store for a given query"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings(query)[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved = []
        for doc, metadata, distance in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            retrieved.append({
                "content": doc,
                "metadata": metadata,
                "score": 1 - distance
            })
        return retrieved


retriever = RAGRetriever(vectorstore, embedding_manager)

results = retriever.retrieve("What is RAG?", top_k=3)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(r['content'][:200])
    print(r['metadata'])
    print("---")

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Score: 0.624
How RAG Works
The RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).
Indexing (Preparation): Documents are split into small sections called chunks. 
{'doc_index': 1, 'content_length': 709, 'source': '../data/text_files/rag_intro.txt'}
---
Score: 0.624
How RAG Works
The RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).
Indexing (Preparation): Documents are split into small sections called chunks. 
{'source': '../data/text_files/rag_intro.txt', 'content_length': 709, 'doc_index': 1}
---
Score: 0.558
Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from external knowledge bases before generating a response
{'source': '../data/text_files/rag_intro.txt', 'doc_index': 0, 'content_length': 527}
---


In [ ]:
# Install Groq client (if not already installed in cell 1)
# already have langchain-groq installed, just need the API key

import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

# Colab-safe way to enter your key without printing it
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.1,
    max_tokens=1024
)

RAG_PROMPT = ChatPromptTemplate.from_template("""Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Use three sentences maximum and keep the answer as concise as possible.

Context:
{context}

Question: {question}

Answer:
""")

def rag_query(question: str, top_k: int = 3, score_threshold: float = 0.3):
    results = retriever.retrieve(question, top_k=top_k)

    # filter out weak matches
    good_results = [r for r in results if r["score"] >= score_threshold]

    if not good_results:
        return {
            "answer": "I don't have enough information to answer that.",
            "sources": []
        }

    context = "\n\n---\n\n".join(r["content"] for r in good_results)
    messages = RAG_PROMPT.format_messages(context=context, question=question)
    response = llm.invoke(messages)

    return {
        "answer": response.content,
        "sources": [
            {"source": r["metadata"].get("source", r["metadata"].get("source_file", "unknown")),
             "score": round(r["score"], 3)}
            for r in good_results
        ]
    }




# Test it
# Ask a real question about your uploaded document
print("RAG Chatbot — ask questions about your document (type 'exit' or 'quit' to stop)\n")

while True:
    question = input("You: ").strip()

    if question.lower() in ["exit", "quit", "q", ""]:
        print("Chat ended.")
        break

    result = rag_query(question)

    print(f"\nBot: {result['answer']}")

    if result["sources"]:
        print("\nSources:")
        for s in result["sources"]:
            print(f"  - {s['source']} (score: {s['score']})")
    print("\n" + "-"*50 + "\n")

RAG Chatbot — ask questions about your document (type 'exit' or 'quit' to stop)

You: quit
Chat ended.


In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
models = client.models.list()

print("Models available to your key:\n")
for m in models.data:
    print("-", m.id)

Models available to your key:

- allam-2-7b
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-arabic-saudi
- openai/gpt-oss-120b
- canopylabs/orpheus-v1-english
- qwen/qwen3.8-27b
- meta-llama/llama-prompt-guard-2-86m
- groq/compound-mini
- openai/gpt-oss-20b
- whisper-large-v3-turbo
- qwen/qwen3.6-27b
- whisper-large-v3
- meta-llama/llama-prompt-guard-2-22m
- groq/compound


In [ ]:
# Debug: see what the retriever actually returns before filtering
raw_results = retriever.retrieve("What is RAG?", top_k=3)
for r in raw_results:
    print(f"Score: {r['score']:.3f} | Metadata: {r['metadata']}")
    print(r['content'][:150])
    print("---")

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Score: 0.624 | Metadata: {'content_length': 709, 'source': '../data/text_files/rag_intro.txt', 'doc_index': 1}
How RAG Works
The RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).
Indexing (Preparation): Docum
---
Score: 0.624 | Metadata: {'content_length': 709, 'source': '../data/text_files/rag_intro.txt', 'doc_index': 1}
How RAG Works
The RAG process works in two main stages: preparation (indexing) and execution (retrieval and generation).
Indexing (Preparation): Docum
---
Score: 0.558 | Metadata: {'content_length': 527, 'doc_index': 0, 'source': '../data/text_files/rag_intro.txt'}
Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves a large language model's accuracy by fetching data from ext
---
